In [ ]:
import tqdm
import requests
import numpy as np
import pandas as pd
import seaborn as sns
import networkx as nx
import plotly.express as px
from bs4 import BeautifulSoup
import matplotlib.pyplot as plt
import pandas_datareader.data as web

import warnings
warnings.filterwarnings('ignore')
import yfinance as yf

In [ ]:
# Load CSV
csv_path = r"C:\Users\goati\prolific_g.o.a.t\holistics_laboratory_of_porlifics\1. FINVIZ\1. Stock_Symbols\stock_symbols.csv"
tickers_df = pd.read_csv(csv_path)


# Normalize column name (handles Symbol, Ticker, etc.)
possible_cols = ["Symbol", "Ticker", "Tickers", "stock", "stocks"]
    
# Find the ticker column
ticker_col = next((col for col in tickers_df.columns if col in possible_cols), None)
if ticker_col is None:
    raise ValueError("No ticker column found in CSV. Check column names.")

# Extract tickers as a Python list
tickers = tickers_df[ticker_col].dropna().unique().tolist()

# Print the number of tickers loaded
print(f"Loaded {len(tickers)} tickers")

In [ ]:
import yfinance as yf
import pandas as pd
import re

# ----------------------------------------------------
# 1. Clean and sort tickers
# ----------------------------------------------------
cleaned_tickers = []

for t in tickers:
    if isinstance(t, str):
        t_clean = re.sub(r'[^A-Za-z0-9.-]', '', t).lstrip('$').strip()
        if t_clean:
            cleaned_tickers.append(t_clean)

cleaned_tickers = sorted(cleaned_tickers)

# ----------------------------------------------------
# 2. Download all tickers at once
# ----------------------------------------------------
combined_df = yf.download(
    tickers=cleaned_tickers,
    period="50y",
    group_by="ticker",
    threads=True,
    auto_adjust=False
)

# ----------------------------------------------------
# 3. Round numeric columns
# ----------------------------------------------------
combined_df = combined_df.round(2)

# ----------------------------------------------------
# 4. Reset index (flatten MultiIndex)
# ----------------------------------------------------
combined_df = combined_df.stack(level=0).reset_index()

# Rename columns for clarity
combined_df.columns = [
    "Date", "Ticker", "Open", "High", "Low", "Close", "Adj Close", "Volume"
    
]

# ----------------------------------------------------
# 5. Sort by Ticker + Date
# ----------------------------------------------------
combined_df = combined_df.sort_values(["Ticker", "Date"])

# ----------------------------------------------------
# 6. Save flattened CSV
# ----------------------------------------------------
combined_df.to_csv("price_action_data.csv", index=False)

print("CSV saved as price_action_data.csv")


In [ ]:
# Set Date as the index
universal_df = combined_df.set_index("Date")

# Sort by Date for clean analytics
universal_df = universal_df.sort_index()

# Save universal_df to CSV
universal_df.to_csv("universal_df.csv", index=True)
print("universal_df.csv has been saved")

# display dataframe
universal_df

In [ ]:
import pandas as pd

df = pd.read_csv("universal_df.csv", parse_dates=["Date"])

# Pivot into price matrix
price_action_data = df.pivot_table(
    index="Date",
    columns="Ticker",
    values="Adj Close"
)

# Sort by date
price_action_data = price_action_data.sort_index()

# Save price_action_data to CSV
price_action_data.to_csv("price_action_data.csv")

# display price_action_data
price_action_data

## **Getting the Price Data in the last 10 years**

In [ ]:
import pandas as pd

# Load flat CSV
df = pd.read_csv("price_action_data.csv", parse_dates=["Date"])

# Convert Date column to pure date (removes timestamp entirely)
df["Date"] = df["Date"].dt.date

# Set Date as index
df = df.set_index("Date").sort_index()

# Compute dynamic 10-year window as pure dates
end = pd.Timestamp.today().date()
start = (pd.Timestamp.today() - pd.DateOffset(years=10)).date()

# Slice last 10 years (now both sides are datetime.date)
last_10yr_price_data = df.loc[start:end]

# Save using the same variable name
last_10yr_price_data.to_csv("last_10yr_price_data.csv")

print("Saved last_10yr_price_data.csv")

last_10yr_price_data


In [ ]:
# Create a copy of the last 10 years of price data
price_data_10yr = last_10yr_price_data.copy()

## **Missing Data due to Index Rebalancing**

In [ ]:
plt.figure(figsize=(16, 8))

sns.heatmap(
    price_data_10yr.T.isnull(),
)

plt.title("Missing Data Heatmap (10-Year Price Matrix)")
plt.xlabel("Date")
plt.ylabel("Ticker")

plt.show()


In [ ]:
# Drop all na values
price_data_cleaned = price_data_10yr.dropna(axis=1) 

In [ ]:
figure = plt.figure(figsize=(16, 8))
sns.heatmap(price_data_cleaned.T.isnull());

## **Getting Yearwise Data**

In [ ]:
def get_annual_price_action_data(data, year):
    start = pd.Timestamp(year, 1, 1).date()
    end = pd.Timestamp(year, 12, 31).date()
    return data.loc[start:end]


In [ ]:
# Getting year wise data of S&P stocks from 2011 to 2020
year_2016 = get_annual_price_action_data(price_data_cleaned, 2016)
year_2017 = get_annual_price_action_data(price_data_cleaned, 2017)
year_2018 = get_annual_price_action_data(price_data_cleaned, 2018)
year_2019 = get_annual_price_action_data(price_data_cleaned, 2019)
year_2020 = get_annual_price_action_data(price_data_cleaned, 2020)
year_2021 = get_annual_price_action_data(price_data_cleaned, 2021)
year_2022 = get_annual_price_action_data(price_data_cleaned, 2022)
year_2023 = get_annual_price_action_data(price_data_cleaned, 2023)
year_2024 = get_annual_price_action_data(price_data_cleaned, 2024)
year_2025 = get_annual_price_action_data(price_data_cleaned, 2025)

In [ ]:
year_2016

In [ ]:
year_2016.shift(1)

## **Computing the Daily Log Returns**

Statistically, **simple stock returns are always assumed to follow a Log Normal distribution**. It is therefore plausible to use properties of the Normal distribution in statistical estimation for Log returns, but not for the simple returns.

Stock Returns analysis is a time series analysis, in which you also take care of stationarity which is normally obtained from Log returns but not from simple returns.

In [ ]:
# Calculating daily log returns by subtracting between two days with the help of shift function
log_returns_2016 = np.log(year_2016.shift(1)) - np.log(year_2016)
log_returns_2017 = np.log(year_2017.shift(1)) - np.log(year_2017)
log_returns_2018 = np.log(year_2018.shift(1)) - np.log(year_2018)
log_returns_2019 = np.log(year_2019.shift(1)) - np.log(year_2019)
log_returns_2020 = np.log(year_2020.shift(1)) - np.log(year_2020)
log_returns_2021 = np.log(year_2021.shift(1)) - np.log(year_2021)
log_returns_2022 = np.log(year_2022.shift(1)) - np.log(year_2022)
log_returns_2023 = np.log(year_2023.shift(1)) - np.log(year_2023)
log_returns_2024 = np.log(year_2024.shift(1)) - np.log(year_2024)
log_returns_2025 = np.log(year_2025.shift(1)) - np.log(year_2025)

## **Computing the Correlation of Returns**

In [ ]:
# Computing adjacency matrix:
return_correlation_2016 = log_returns_2016.corr()
return_correlation_2017 = log_returns_2017.corr()
return_correlation_2018 = log_returns_2018.corr()
return_correlation_2019 = log_returns_2019.corr()
return_correlation_2020 = log_returns_2020.corr()
return_correlation_2021 = log_returns_2021.corr()
return_correlation_2022 = log_returns_2022.corr()
return_correlation_2023 = log_returns_2023.corr()
return_correlation_2024 = log_returns_2024.corr()
return_correlation_2025 = log_returns_2025.corr()


In [ ]:
figure, axes = plt.subplots(5, 2, figsize=(30, 30))
sns.heatmap(return_correlation_2016, ax=axes[0, 0]);
sns.heatmap(return_correlation_2017, ax=axes[0, 1]);
sns.heatmap(return_correlation_2018, ax=axes[1, 0]);
sns.heatmap(return_correlation_2019, ax=axes[1, 1]);
sns.heatmap(return_correlation_2020, ax=axes[2, 0]);
sns.heatmap(return_correlation_2021, ax=axes[2, 1]);
sns.heatmap(return_correlation_2022, ax=axes[3, 0]);
sns.heatmap(return_correlation_2023, ax=axes[3, 1]);
sns.heatmap(return_correlation_2024, ax=axes[4, 0]);
sns.heatmap(return_correlation_2025, ax=axes[4, 1]);

## **Creating Graphs**

In [ ]:
graph_2016 = nx.Graph(return_correlation_2016)
graph_2017 = nx.Graph(return_correlation_2017)
graph_2018 = nx.Graph(return_correlation_2018)
graph_2019 = nx.Graph(return_correlation_2019)
graph_2020 = nx.Graph(return_correlation_2020)
graph_2021 = nx.Graph(return_correlation_2021)
graph_2022 = nx.Graph(return_correlation_2022)
graph_2023 = nx.Graph(return_correlation_2023)
graph_2024 = nx.Graph(return_correlation_2024)
graph_2025 = nx.Graph(return_correlation_2025)

In [ ]:
figure = plt.figure(figsize=(22, 10))
nx.draw_networkx(graph_2020, with_labels=False)

## **Filtering Graphs using MST**

**MST - Minimum Spanning Tree**

A minimum spanning tree (MST) or minimum weight spanning tree is a subset of the edges of a connected, edge-weighted undirected graph that connects all the vertices together, without any cycles and with the minimum possible total edge weight.That is, it is a spanning tree whose sum of edge weights is as small as possible.

**MST** is one of the popular techniques to eliminate the redundancies and noise and meanwhile maintain the significant links in the network.

While removing redundancy and noise in the data using MST, we might lose some information as well.

You can find more on MST [here](https://visualgo.net/en/mst)

In [ ]:
# Calculating distance matrix from correlation matrix using the formula: distance = sqrt(2 * (1 - correlation))
distance_2016 = np.sqrt(2 * (1 - return_correlation_2016))
distance_2017 = np.sqrt(2 * (1 - return_correlation_2017))
distance_2018 = np.sqrt(2 * (1 - return_correlation_2018))
distance_2019 = np.sqrt(2 * (1 - return_correlation_2019))
distance_2020 = np.sqrt(2 * (1 - return_correlation_2020))
distance_2021 = np.sqrt(2 * (1 - return_correlation_2021))
distance_2022 = np.sqrt(2 * (1 - return_correlation_2022))
distance_2023 = np.sqrt(2 * (1 - return_correlation_2023))
distance_2024 = np.sqrt(2 * (1 - return_correlation_2024))
distance_2025 = np.sqrt(2 * (1 - return_correlation_2025))

In [ ]:
# Create NetworkX graphs from the distance matrices
distance_2016_graph = nx.Graph(distance_2016)
distance_2017_graph = nx.Graph(distance_2017)
distance_2018_graph = nx.Graph(distance_2018)
distance_2019_graph = nx.Graph(distance_2019)
distance_2020_graph = nx.Graph(distance_2020)
distance_2021_graph = nx.Graph(distance_2021)
distance_2022_graph = nx.Graph(distance_2022)
distance_2023_graph = nx.Graph(distance_2023)
distance_2024_graph = nx.Graph(distance_2024)
distance_2025_graph = nx.Graph(distance_2025)

In [ ]:
# Filter the graphs to retain only the minimum spanning tree (MST) edges
graph_2016_filtered = nx.minimum_spanning_tree(distance_2016_graph)
graph_2017_filtered = nx.minimum_spanning_tree(distance_2017_graph)
graph_2018_filtered = nx.minimum_spanning_tree(distance_2018_graph)
graph_2019_filtered = nx.minimum_spanning_tree(distance_2019_graph)
graph_2020_filtered = nx.minimum_spanning_tree(distance_2020_graph)
graph_2021_filtered = nx.minimum_spanning_tree(distance_2021_graph)
graph_2022_filtered = nx.minimum_spanning_tree(distance_2022_graph)
graph_2023_filtered = nx.minimum_spanning_tree(distance_2023_graph)
graph_2024_filtered = nx.minimum_spanning_tree(distance_2024_graph)
graph_2025_filtered = nx.minimum_spanning_tree(distance_2025_graph)

##### The MST method is used to filter out the network graph in each window so as to eliminate the redundancies and noise, and still maintain significant links.

In [ ]:
# Visualize the filtered graphs for each year
figure, axes = plt.subplots(10, 1, figsize=(24, 120))
nx.draw_networkx(graph_2016_filtered, with_labels=False, ax=axes[0])
nx.draw_networkx(graph_2017_filtered, with_labels=False, ax=axes[1])
nx.draw_networkx(graph_2018_filtered, with_labels=False, ax=axes[2])
nx.draw_networkx(graph_2019_filtered, with_labels=False, ax=axes[3])
nx.draw_networkx(graph_2020_filtered, with_labels=False, ax=axes[4])
nx.draw_networkx(graph_2021_filtered, with_labels=False, ax=axes[5])
nx.draw_networkx(graph_2022_filtered, with_labels=False, ax=axes[6])
nx.draw_networkx(graph_2023_filtered, with_labels=False, ax=axes[7])
nx.draw_networkx(graph_2024_filtered, with_labels=False, ax=axes[8])
nx.draw_networkx(graph_2025_filtered, with_labels=False, ax=axes[9])

## **Computing Graph Statistics over Time**

In [ ]:
average_degree_connectivity = []
average_shortest_path_length = []
year = [2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]

for graph in [graph_2016_filtered, graph_2017_filtered, graph_2018_filtered, graph_2019_filtered, graph_2020_filtered,
             graph_2021_filtered, graph_2022_filtered, graph_2023_filtered, graph_2024_filtered, graph_2025_filtered]:
    average_shortest_path_length.append(nx.average_shortest_path_length(graph))

In [ ]:
figure = plt.figure(figsize=(22, 7))
sns.lineplot(x='year', y='average_shortest_path_length',
             data=pd.DataFrame({'year': year, 'average_shortest_path_length': average_shortest_path_length}));

From 2016 to 2025, the line fluctuates noticeably. The pattern suggests:

Peaks around 2016, 2020, and 2024  
These years show higher average shortest path length, meaning the network temporarily became less efficient or more dispersed.

Dips around 2018 and 2022  
These years show lower values, meaning the network became more tightly connected.

This oscillation implies your network undergoes periodic structural shifts—sometimes clustering tightly, sometimes spreading out.

## **Portfolio Construction**

In [ ]:
log_returns_2016_till_2025 = np.log(price_data_cleaned.shift(1)) - np.log(price_data_cleaned)
return_correlation_2016_till_2025 = log_returns_2016_till_2025.corr()

In [ ]:
figure = plt.figure(figsize=(24, 8))
sns.heatmap(return_correlation_2016_till_2025);

In [ ]:
distance_2016_till_2025 = np.sqrt(2 * (1 - return_correlation_2016_till_2025))
distance_2016_till_2025_graph = nx.Graph(distance_2016_till_2025)
distance_2016_till_2025_graph_filtered = nx.minimum_spanning_tree(distance_2016_till_2025_graph)

In [ ]:
figure = plt.figure(figsize=(24, 8))
nx.draw_kamada_kawai(distance_2016_till_2025_graph_filtered, with_labels=False)

In [ ]:
degree_centrality = nx.degree_centrality(distance_2016_till_2025_graph_filtered)
closeness_centrality = nx.closeness_centrality(distance_2016_till_2025_graph_filtered)
betweenness_centrality = nx.betweenness_centrality(distance_2016_till_2025_graph_filtered)
eigenvector_centrality=nx.eigenvector_centrality_numpy(distance_2016_till_2025_graph_filtered)

In [ ]:
keys = []
values = []

for key, value in degree_centrality.items():
    keys.append(key)
    values.append(value)

dc_data = pd.DataFrame({'stocks': keys, 'degree_centrality': values}).sort_values('degree_centrality', ascending=False)
px.bar(data_frame=dc_data, x='stocks', y='degree_centrality', template='plotly_dark')

**Degree centrality** is the simplest centrality measure. It
defines the relative significance of a stock in terms of the
**number of edges incident upon it.** The stocks with the high
scores will influence the behavior of many other
stocks which are directly connected to it.

Based on this
measure, **PH has the highest number of edges with other
stocks and hence the highest degree centrality.**

In [ ]:
keys = []
values = []

for key, value in closeness_centrality.items():
    keys.append(key)
    values.append(value)

cc_data = pd.DataFrame({'stocks': keys, 'closeness_centrality': values}).sort_values('closeness_centrality',
                                                                                       ascending=False)
px.bar(data_frame=cc_data, x='stocks', y='closeness_centrality', template='plotly_dark')

**Closeness centrality** also involves the shortest path
between all possible pairs of stocks on a network.

It is
defined as the average number of shortest paths between a
stock and all other stocks reachable from it.

In [ ]:
keys = []
values = []

for key, value in betweenness_centrality.items():
    keys.append(key)
    values.append(value)

bc_data = pd.DataFrame({'stocks': keys, 'betweenness_centrality': values}).sort_values('betweenness_centrality',
                                                                                       ascending=False)
px.bar(data_frame=bc_data, x='stocks', y='betweenness_centrality', template='plotly_dark')

**Betweenness centrality** is the sum of the fraction of all possible shortest paths between any stocks that pass through a stock. It is used to **quantify the control of a stock on information flow in the network.**

So, the stock with the highest score is considered a significant stock in terms of its role in coordinating the information among stocks.

In [ ]:
# Both the degree centrality and the betweenness centrality has been computed above

# Distance on degree criterion
distance_degree_criteria = {}
node_with_largest_degree_centrality = max(dict(degree_centrality), key=dict(degree_centrality).get)
for node in distance_2016_till_2025_graph_filtered.nodes():
    distance_degree_criteria[node] = nx.shortest_path_length(distance_2016_till_2025_graph_filtered, node,
                                                             node_with_largest_degree_centrality)

# Distance on correlation criterion
distance_correlation_criteria = {}
sum_correlation = {}

for node in distance_2016_till_2025_graph_filtered.nodes():
    neighbors = nx.neighbors(distance_2016_till_2025_graph_filtered, node)
sum_correlation[node] = sum(return_correlation_2016_till_2025[node][neighbor] for neighbor in neighbors)

node_with_highest_correlation = max(sum_correlation, key=sum_correlation.get)

for node in distance_2016_till_2025_graph_filtered.nodes():
    distance_correlation_criteria[node] = nx.shortest_path_length(distance_2016_till_2025_graph_filtered, node,
                                                             node_with_highest_correlation)

# distance on distance criterion
distance_distance_criteria = {}
mean_distance = {}

for node in distance_2016_till_2025_graph_filtered.nodes():
    nodes = list(distance_2016_till_2025_graph_filtered.nodes())
    nodes.remove(node)
    distance_distance = [nx.shortest_path_length(distance_2016_till_2025_graph_filtered, node, ns) for ns in nodes]
    mean_distance[node] = np.mean(distance_distance)

node_with_minimum_mean_distance = min(mean_distance, key=mean_distance.get)

for node in distance_2016_till_2025_graph_filtered.nodes():
    distance_distance_criteria[node] = nx.shortest_path_length(distance_2016_till_2025_graph_filtered, node,
                                                             node_with_minimum_mean_distance)

**Distance refers to the smallest length from a node to the central node of the network**.

Three types of distances:

**1. Distance on degree criterion** (Ddegree), the central node is the one that has the largest degree.

**2. Distance on correlation criterion** (Dcorrelation), the central node is the one with the highest value of the sum of correlation coefficients with its neighbors.

**3. Distance on distance criterion** (Ddistance), the central node is the one that produces the lowest value for the mean distance.

The three types of definitions of central node are introduced to reduce the error caused by a single
method.

In [ ]:
node_stats = pd.DataFrame.from_dict(dict(degree_centrality), orient='index')
node_stats.columns = ['degree_centrality']
node_stats['betweenness_centrality'] = betweenness_centrality.values()

node_stats['average_centrality'] = 0.5 * (node_stats['degree_centrality'] + node_stats['betweenness_centrality'])

node_stats['distance_degree_criteria'] = distance_degree_criteria.values()
node_stats['distance_correlation_criteria'] = distance_correlation_criteria.values()
node_stats['distance_distance_criteria'] = distance_distance_criteria.values()
node_stats['average_distance'] = (node_stats['distance_degree_criteria'] + node_stats['distance_correlation_criteria'] +
                                  node_stats['distance_distance_criteria']) / 3



In [ ]:
node_stats

We use the parameters defined above to select the portfolios.

**The nodes with the largest 10% of degree or betweenness centrality are chosen to be in the central portfolio.**

**The nodes whose degree equals to 1 or betweenness centrality equals to 0 are chosen to be in the peripheral portfolio.**


Similarly, we define the node's ranking in the top 10% of distance as the stocks of the peripheral portfolios, and the bottom 10% as the stocks of the central portfolios.

The central portfolios and peripheral portfolios represent two opposite sides of correlation and agglomeration. Generally speaking, central stocks play a vital role in the market and impose a strong influence on other stocks. On the other hand, the correlations between peripheral stocks are weak and contain much more noise than those of the central stocks.

In [ ]:
central_stocks = node_stats.sort_values('average_centrality', ascending=False).head(15)
central_portfolio = [stock for stock in central_stocks.index.values]

In [ ]:
peripheral_stocks = node_stats.sort_values('average_distance', ascending=False).head(15)
peripheral_portfolio = [stock for stock in peripheral_stocks.index.values]

In [ ]:
central_stocks

In [ ]:
peripheral_stocks

### **Selecting the top 15 stocks for both Central Stocks and Peripheral Stocks**

In [ ]:
color = []

for node in distance_2016_till_2025_graph_filtered:
    if node in central_portfolio:
        color.append('red')

    elif node in peripheral_portfolio:
        color.append('green')

    else:
        color.append('blue')

In [ ]:
figure = plt.figure(figsize=(24, 8))
nx.draw_kamada_kawai(distance_2016_till_2025_graph_filtered, with_labels=False, node_color=color)

**The red stocks are the central portfolio stocks, and the green ones are the peripheral portfolio stocks.**

## **Performance Evalutation**

In [ ]:
import pandas as pd

# ----------------------------------------------------
# 1. Load the CSV
# ----------------------------------------------------
df = pd.read_csv("universal_df.csv")

# Ensure Date is datetime
df["Date"] = pd.to_datetime(df["Date"])

# ----------------------------------------------------
# 2. Filter for 2021 only
# ----------------------------------------------------
price_action_data_2021 = df[
    (df["Date"] >= "2021-01-01") &
    (df["Date"] <= "2021-12-31")
]

# ----------------------------------------------------
# 3. Extract Adj Close only
# ----------------------------------------------------
price_action_data_2021 = price_action_data_2021[["Date", "Ticker", "Adj Close"]]

# ----------------------------------------------------
# 4. Pivot: Date becomes index, tickers become columns
# ----------------------------------------------------
price_action_data_2021 = price_action_data_2021.pivot(
    index="Date",
    columns="Ticker",
    values="Adj Close"
)

# ----------------------------------------------------
# 5. Save CSV
# ----------------------------------------------------
price_action_data_2021.to_csv("price_action_data_2021.csv")

print("CSV saved as price_action_data_2021.csv")


In [ ]:
import yfinance as yf
import pandas as pd
import re

# ----------------------------------------------------
# 1. Define tickers (DIA, SPY, NDAQ)
# ----------------------------------------------------
tickers = ["DIA", "SPY", "NDAQ"]

# Clean and sort tickers
cleaned_tickers = []

for t in tickers:
    if isinstance(t, str):
        t_clean = re.sub(r'[^A-Za-z0-9.-]', '', t).lstrip('$').strip()
        if t_clean:
            cleaned_tickers.append(t_clean)

cleaned_tickers = sorted(cleaned_tickers)

# ----------------------------------------------------
# 2. Download all tickers at once
# ----------------------------------------------------
NASDAQ_DIA_SPY_price_action = yf.download(
    tickers=cleaned_tickers,
    period="50y",
    group_by="ticker",
    threads=True,
    auto_adjust=False
)

# ----------------------------------------------------
# 3. Round numeric columns
# ----------------------------------------------------
NASDAQ_DIA_SPY_price_action = NASDAQ_DIA_SPY_price_action.round(2)

# ----------------------------------------------------
# 4. Reset index (flatten MultiIndex)
# ----------------------------------------------------
NASDAQ_DIA_SPY_price_action = (
    NASDAQ_DIA_SPY_price_action
    .stack(level=0)
    .reset_index()
)

# Rename columns for clarity
NASDAQ_DIA_SPY_price_action.columns = [
    "Date", "Ticker", "Open", "High", "Low", "Close", "Adj Close", "Volume"
]

# ----------------------------------------------------
# 5. Sort by Ticker + Date
# ----------------------------------------------------
NASDAQ_DIA_SPY_price_action = NASDAQ_DIA_SPY_price_action.sort_values(
    ["Ticker", "Date"]
)

# ----------------------------------------------------
# 6. Save flattened CSV
# ----------------------------------------------------
NASDAQ_DIA_SPY_price_action.to_csv(
    "NASDAQ_DIA_SPY_price_action.csv",
    index=False
)

print("CSV saved as NASDAQ_DIA_SPY_price_action.csv")


In [ ]:
# ----------------------------------------------------
# 1. Load CSV
# ----------------------------------------------------
df = pd.read_csv("NASDAQ_DIA_SPY_price_action.csv")

# Ensure Date is datetime
df["Date"] = pd.to_datetime(df["Date"])

# ----------------------------------------------------
# 2. Filter for 2021 only (***** Think of a better universal solution *****)
# ----------------------------------------------------
price_data_2021 = df[
    (df["Date"] >= "2021-01-01") &
    (df["Date"] <= "2021-12-31")
]

# ----------------------------------------------------
# 3. Extract Adj Close only
# ----------------------------------------------------
price_data_2021 = price_data_2021[["Date", "Ticker", "Adj Close"]]

# ----------------------------------------------------
# 4. Save CSV
# ----------------------------------------------------
price_data_2021.to_csv("NASDAQ_DIA_SPY_price_data_2021.csv", index=False)

print("CSV saved as NASDAQ_DIA_SPY_price_data_2021.csv")


In [ ]:
NASDAQ_DIA_SPY_price_action

In [ ]:
from pandas_datareader import data as web
import pandas as pd

# ----------------------------------------------------
# Retrieve ALL available historical data from FRED
# ----------------------------------------------------
fred_index_data = web.DataReader(
    ['SP500', 'NASDAQCOM', 'DJIA'],
    'fred'
)

# Save to CSV
fred_index_data.to_csv("fred_index_data.csv")

print("CSV saved as fred_index_data.csv")


In [ ]:
import pandas as pd

# ----------------------------------------------------
# 1. Load the CSV you previously created
# ----------------------------------------------------
df = pd.read_csv("NASDAQ_DIA_SPY_price_action.csv")

# Ensure Date is datetime
df["Date"] = pd.to_datetime(df["Date"])

# ----------------------------------------------------
# 2. Filter for 2021 only
# ----------------------------------------------------
NASDAQ_DIA_SPY_price_data_2021 = df[
    (df["Date"] >= "2021-01-01") &
    (df["Date"] <= "2021-12-31")
]

# ----------------------------------------------------
# 3. Extract Adj Close only
# ----------------------------------------------------
NASDAQ_DIA_SPY_price_data_2021 = NASDAQ_DIA_SPY_price_data_2021[
    ["Date", "Ticker", "Adj Close"]
]

# ----------------------------------------------------
# 4. Pivot so Date is index and tickers are columns
# ----------------------------------------------------
NASDAQ_DIA_SPY_price_data_2021 = NASDAQ_DIA_SPY_price_data_2021.pivot(
    index="Date",
    columns="Ticker",
    values="Adj Close"
)

# ----------------------------------------------------
# 5. Save CSV
# ----------------------------------------------------
NASDAQ_DIA_SPY_price_data_2021.to_csv(
    "NASDAQ_DIA_SPY_price_data_2021.csv"
)

print("CSV saved as NASDAQ_DIA_SPY_price_data_2021.csv")


In [ ]:
# ----------------------------------------------------
# 1. Remove NA values from price_action_data_2021
# ----------------------------------------------------
price_action_data_2021 = price_action_data_2021.dropna(axis=1)

# ----------------------------------------------------
# 2. Remove NA values from NASDAQ_DIA_SPY_price_data_2021
# ----------------------------------------------------
NASDAQ_DIA_SPY_price_data_2021 = NASDAQ_DIA_SPY_price_data_2021.dropna(axis=1)


In [ ]:
price_action_data_2021

In [ ]:
NASDAQ_DIA_SPY_price_data_2021

In [ ]:
price_action_data_2021 = price_action_data_2021['2021-01-04':]

In [ ]:
amount = 100000

central_portfolio_value = pd.DataFrame()
for stock in central_portfolio:
    central_portfolio_value[stock] = price_action_data_2021[stock]

portfolio_unit = central_portfolio_value.sum(axis=1)[0]
share = amount / portfolio_unit
central_portfolio_value = central_portfolio_value.sum(axis=1) * share

peripheral_portfolio_value = pd.DataFrame()
for stock in peripheral_portfolio:
    peripheral_portfolio_value[stock] = price_action_data_2021[stock]

portfolio_unit = peripheral_portfolio_value.sum(axis=1)[0]
share = amount / portfolio_unit
peripheral_portfolio_value = peripheral_portfolio_value.sum(axis=1) * share

In [ ]:
Value_NASDAQ_DIA_SPY_price_data_2021 = NASDAQ_DIA_SPY_price_data_2021 * (amount / NASDAQ_DIA_SPY_price_data_2021.iloc[0])

In [ ]:
all_portfolios = Value_NASDAQ_DIA_SPY_price_data_2021
all_portfolios['central_portfolio'] = central_portfolio_value.values
all_portfolios['peripheral_portfolio'] = peripheral_portfolio_value.values

In [ ]:
all_portfolios

In [ ]:
figure, ax = plt.subplots(figsize=(16, 8))

# Plot each series
# Plot the NASDAQ, DIA, and S&P
dia_line = ax.plot(all_portfolios['DIA'], label='DOW JONES')
ndaq_line = ax.plot(all_portfolios['NDAQ'], label='NASDAQ')
spy_line = ax.plot(all_portfolios['SPY'], label='S&P 500')

# Plot the Central and Peripheral portfolio
central_portfolio_line = ax.plot(all_portfolios['central_portfolio'],label='Central Portfolio')
peripheral_portfolio_line = ax.plot(all_portfolios['peripheral_portfolio'], label='Peripheral Portfolio')

# Legend + formatting
ax.legend(loc='upper left')
ax.set_title("2021 Portfolio Comparison", fontsize=18)
ax.set_xlabel("Date", fontsize=14)
ax.set_ylabel("Portfolio Value ($)", fontsize=14)

plt.show()


**Key Takeaways:**

In 2021, the Central Portfolio significantly outperformed both the Peripheral Portfolio and traditional benchmarks (SPY, DIA, NDAQ).

This reinforces the idea that:

- Central stocks excel in stable or bullish markets

- Peripheral stocks provide defensive characteristics during crisis periods

- These dynamics allows investors to rebalance intelligently based on market regime by using network analysis as a structural guide rather than relying solely on traditional metrics.

**Network analysis divides stocks into two structural categories:** (Central vs. Peripheral Portfolio Behavior)

**Central Portfolio**
- Highly connected stocks within the market network

- Strongly correlated with broad market movements

- Typically large‑cap, liquid, widely held names

- Benefit from synchronized market rallies

**Peripheral Portfolio**
- Weakly connected stocks

- Lower correlation with the rest of the market

- More insulated from systemic shocks

- Tend to show relative strength during crisis periods

This structural difference explains why each portfolio behaves differently depending on the macro environment.

**What the 2021 Graph Shows**

- The plot clearly demonstrates that 2021 was a favorable environment for Central Portfolio stocks:

- The Central Portfolio shows strong, consistent upward movement.

- It outperforms SPY, DIA, and NDAQ for most of the year.

This aligns with the post‑pandemic recovery, high liquidity, and broad market participation seen in 2021.

**In contrast:**

- The Peripheral Portfolio lags behind the benchmarks.

- It shows weaker momentum and lower participation in market rallies.

This is expected in stable or bullish regimes, where peripheral stocks do not benefit as much from synchronized market strength.